# Kata 29 — Confidence Calibration y Stratified Sampling

> Spec: [`specs/029-confidence-calibration/spec.md`](../../specs/029-confidence-calibration/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json, random

client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Confianza self-reported del modelo está mal calibrada (suele
sobreestimar). Calibrarla contra un labeled validation set permite
enrutar trabajo: high-confidence → automatización + stratified sampling;
low-confidence → revisión humana.

## 2. Por qué importa

"97 % accuracy global" puede ocultar 60 % en un tipo específico de doc.
Stratified sampling sobre high-confidence detecta nuevos modos de error
que el validation set viejo no captura.

## 3. Field-level confidence en extracciones

In [ ]:
EXTRACT_CONF = {
    "name": "extract_invoice_with_conf",
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_id": {"type": "string"},
            "invoice_id_confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "total_amount": {"type": ["number", "null"]},
            "total_amount_confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "doc_type": {"type": "string", "enum": ["invoice", "receipt", "statement", "unclear"]},
        },
        "required": ["invoice_id", "invoice_id_confidence", "total_amount", "total_amount_confidence", "doc_type"],
    },
}

def extract(client, doc_text):
    resp = client.messages.create(
        system=(
            "Extrae la factura. Para cada campo, anota tu confianza (0-1) basada en cuán claro está el "
            "dato en la fuente. Confianza alta = el campo aparece textual; baja = inferido o ambiguo."
        ),
        messages=[{"role":"user","content": doc_text}],
        tools=[EXTRACT_CONF],
        tool_choice={"type":"tool","name":"extract_invoice_with_conf"},
    )
    return next(b.input for b in resp.content if b.type == "tool_use")

DOCS = [
    ("invoice_clear", "Factura INV-100. Total: $1,234.56. Cliente: ACME."),
    ("invoice_messy", "Acuerdo entre partes... fecha agosto 2025... aproximadamente $3500."),
    ("receipt_partial", "Recibo No. 88-A. Pagado en efectivo."),
]

extractions = []
for doc_id, text in DOCS:
    e = extract(client, text)
    e["_doc_id"] = doc_id
    extractions.append(e)
    print(f"{doc_id}:")
    print(json.dumps(e, indent=2, ensure_ascii=False))
    print()

### 3.1 Calibración contra labeled set

In [ ]:
# Labeled "ground truth" para los docs (en producción vendría de etiquetadores)
TRUTH = {
    "invoice_clear":   {"invoice_id": "INV-100", "total_amount": 1234.56},
    "invoice_messy":   {"invoice_id": None,     "total_amount": None},  # ambiguo, no extraíble
    "receipt_partial": {"invoice_id": "88-A",   "total_amount": None},
}

def calibrate(extractions, truth):
    """Devuelve mapping confidence_bucket -> (n, accuracy)."""
    buckets = {(0.0, 0.6): [], (0.6, 0.85): [], (0.85, 1.01): []}
    for e in extractions:
        for field in ("invoice_id", "total_amount"):
            conf = e.get(f"{field}_confidence", 0)
            actual = e.get(field)
            expected = truth[e["_doc_id"]][field]
            for (lo, hi), lst in buckets.items():
                if lo <= conf < hi:
                    lst.append(actual == expected)
    return {f"{lo}-{hi}": (len(b), sum(b) / len(b) if b else None) for (lo, hi), b in buckets.items()}

cal = calibrate(extractions, TRUTH)
print("calibración por bucket de confianza:")
for k, (n, acc) in cal.items():
    print(f"  conf {k}: n={n}, accuracy={acc}")

### 3.2 Stratified sampling sobre high-confidence

In [ ]:
def stratified_sample(extractions, n_per_bucket=2):
    by_type = {}
    for e in extractions:
        by_type.setdefault(e["doc_type"], []).append(e)
    sample = []
    for doc_type, items in by_type.items():
        # Filtramos high-confidence por algún campo
        high = [i for i in items if i.get("invoice_id_confidence", 0) >= 0.85]
        if high:
            sample.extend(random.sample(high, min(n_per_bucket, len(high))))
    return sample

# Con 3 docs es chico, pero ilustra el patrón
sample = stratified_sample(extractions)
print("muestra estratificada (high-conf) para revisión humana:")
for s in sample:
    print(f"  {s['_doc_id']}: invoice_id={s['invoice_id']} (conf {s['invoice_id_confidence']})")

## 4. Anti-patrón — confianza raw como filtro de auto-aprobación

In [ ]:
def naive_auto_approve(extractions, threshold=0.85):
    auto = [e for e in extractions if e.get("total_amount_confidence", 0) >= threshold]
    review = [e for e in extractions if e.get("total_amount_confidence", 0) < threshold]
    return auto, review

auto, review = naive_auto_approve(extractions)
print(f"auto-approve: {len(auto)}; humano: {len(review)}")
for e in auto:
    truth = TRUTH[e["_doc_id"]]["total_amount"]
    actual = e["total_amount"]
    flag = "" if actual == truth else "  ← ¡ERROR auto-aprobado!"
    print(f"  {e['_doc_id']}: total={actual} truth={truth}{flag}")

Confianza raw alta sin calibrar puede aprobar errores. El stratified
sampling sobre high-confidence es la red de seguridad.

## 5. Argumento de certificación

- **Confianza raw ≠ probabilidad real** — siempre calibrar contra labeled
  set.
- **Stratified random sampling** sobre high-confidence (no random simple)
  para detectar nuevos modos de error.
- **Accuracy desglosada** por document_type y por field; agregada miente.
- Conexión: Kata 16 (low-conf → escalación) y Kata 20 (provenance permite
  trazar accuracy a la fuente).

## 6. Auto-evaluación

**1. Mi modelo dice `confidence: 0.95` en una extracción. ¿Puedo
automatizar?**

No, no sin calibrar primero. Si el bucket [0.95, 1.0] tiene accuracy
empírica de 99 % en validation, sí. Si tiene 80 %, automatizar = errores
silenciosos.

**2. ¿Cómo construyo un labeled validation set sin pagar mucho a humanos?**

Stratified sampling intencional sobre tus extracciones existentes
(buckets de confianza × document_type), n=50-200, etiquetar a mano. Es
lo mínimo para una primera calibración.

**3. ¿Por qué 97 % global puede ocultar problemas reales?**

Si el 5 % de los docs son "tipo X" y el modelo acierta 50 % en ese tipo,
el global se mantiene cerca de 97 % pero los errores se concentran en X.
Reporting por document_type hace visible el problema.